# Code

In [ ]:
#1. Import Libraries
import pandas as pd
from IPython.display import display

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier

from xgboost import XGBClassifier


In [2]:
seed = 21

In [3]:
# 2. Load Dataset

data = load_wine()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print("Tamaño de X:", X.shape)
print("Tamaño de y:", y.shape)

X.head()

Tamaño de X: (178, 13)
Tamaño de y: (178,)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [4]:
# 3. Divide Dataset in train/test

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=seed,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (142, 13)
Prueba: (36, 13)


In [ ]:
# 4. Create 5 Models and Evaluate

modelos = {
    "Decision Tree": DecisionTreeClassifier(random_state=seed),
    "Bagging Classifier": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=seed),
        n_estimators=50,
        random_state=seed
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=seed
    ),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=seed),
        n_estimators=100,
        learning_rate=0.5,
        random_state=seed
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=seed,
        eval_metric="mlogloss"
    )
}

modelos = {nombre: make_pipeline(StandardScaler(), modelo) for nombre, modelo in modelos.items()}

resultados = []
cv_resultados = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro"),
        "Recall": recall_score(y_test, y_pred, average="macro"),
        "F1-score": f1_score(y_test, y_pred, average="macro")
    })

    acc_cv = cross_val_score(modelo, X, y, cv=cv, scoring="accuracy").mean()
    f1_cv = cross_val_score(modelo, X, y, cv=cv, scoring="f1_macro").mean()

    cv_resultados.append({
        "Modelo": nombre,
        "Accuracy CV": acc_cv,
        "F1-score CV": f1_cv
    })


df_resultados = pd.DataFrame(resultados)
df_cv_resultados = pd.DataFrame(cv_resultados)

display(df_resultados.sort_values(by="F1-score", ascending=False))
display(df_cv_resultados.sort_values(by="F1-score CV", ascending=False))

In [7]:
# 6. Order results by F1-Score

df_resultados.sort_values(by="F1-score", ascending=False)

,Modelo,Accuracy,Precision,Recall,F1-score
3,AdaBoost,1.000000,1.000000,1.000000,1.000000
2,Random Forest,0.972222,0.969697,0.976190,0.971781
1,Bagging Classifier,0.972222,0.969697,0.976190,0.971781
4,XGBoost,0.972222,0.969697,0.976190,0.971781
0,Decision Tree,0.944444,0.945887,0.948413,0.945825


# Explanation

The same five machine learning classification models are compared: Decision Tree, Bagging Classifier, Random Forest, AdaBoost, and XGBoost. In this case the Wine Recognition dataset from sklearn was used, so it was not necessary to create a dataset manually or load the information from external repositories. The goal is to predict the cultivar (one of three wine classes) of an Italian wine sample from 13 chemical features such as alcohol, magnesium, color intensity, and proline. Because the target has three classes instead of two, the precision, recall, and F1-score were computed with `average="macro"` to weight each class equally. The data was split into training and testing sets using the 80/20 distribution with stratification, and each model was trained and evaluated using accuracy, precision, recall, and F1-score.

The best model was AdaBoost, reaching a perfect 1.00 across all metrics on the test set; the ensemble methods (Bagging, Random Forest, and XGBoost) also performed strongly with an F1-score around 0.97, while the single Decision Tree was the weakest at 0.95. A perfect score on such a small test set (only 36 samples) is a warning sign rather than a guarantee of generalization, so it is critical to apply cross-validation and additional regularization to confirm the results are not the product of an easy split or overfitting.